# 04 — Run it six ways

Copy the working functions from notebook 03 into the cells below. Yes, copy-paste. That's fine.

**The one discipline that matters:** before recording any numbers, hit *Restart kernel → Run
all* and let this run top to bottom. If the numbers only appear when cells are run in some
particular order, they aren't numbers.

The response cache makes that cheap — a second full run costs zero API calls.

In [ ]:
import os, sys, json, re
from pathlib import Path

# works locally and in Colab
for candidate in (Path.cwd(), Path.cwd().parent, Path("/content/pa-appeal")):
    if (candidate / "data").exists():
        os.chdir(candidate)
        break
ROOT = Path.cwd()
print("working from:", ROOT)


In [ ]:
# Colab: put the key in the secrets panel (key icon on the left), named GEMINI_API_KEY
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except ImportError:
    pass   # locally: export GEMINI_API_KEY=... before launching jupyter

assert os.environ.get("GEMINI_API_KEY"), "no API key found"
print("key loaded")


In [ ]:
policies = {p.stem: p.read_text() for p in sorted(Path("data/policies").glob("*.md"))}
criteria  = json.load(open("data/criteria.json"))
cases     = json.load(open("data/cases.json"))
Path("data/results").mkdir(parents=True, exist_ok=True)


## Paste in from notebook 03\n\nchunkers, build_index, search, rerank, ask/ask_json, decide, normalize, check_quote, abstain

In [ ]:
# <- paste here


## The six configs

Each row differs from the one above by exactly one setting. That's what makes it an experiment
instead of six unrelated runs.

In [ ]:
CONFIGS = [
    {"name": "row0_context_only", "chunking": None,       "retrieval": None,     "verify": True,  "abstain": False},
    {"name": "row1_naive",        "chunking": "fixed",    "retrieval": "dense",  "verify": False, "abstain": False},
    {"name": "row2_structure",    "chunking": "criteria", "retrieval": "dense",  "verify": False, "abstain": False},
    {"name": "row3_hybrid",       "chunking": "criteria", "retrieval": "hybrid", "verify": False, "abstain": False},
    {"name": "row4_rerank",       "chunking": "criteria", "retrieval": "rerank", "verify": False, "abstain": False},
    {"name": "row5_full",         "chunking": "criteria", "retrieval": "rerank", "verify": True,  "abstain": True},
]


Row 0 sends the whole policy with no retrieval. It's required — L33718 fits in a context
window, so the obvious question is "why build retrieval at all?" I run it and publish the
answer. **If it wins, I say so**, and note that it burns far more tokens per case and doesn't
scale past one policy.

In [ ]:
from tqdm.auto import tqdm

def run_one_case(case, cfg, index):
    """-> list of decision dicts with gold label, predicted label, quote status."""
    raise NotImplementedError("wire together the pieces pasted above")

def run_config(cfg):
    index = None if cfg["retrieval"] is None else build_index(
        chunk_by_criteria(criteria, policies) if cfg["chunking"] == "criteria"
        else [c for d, t in policies.items() for c in chunk_fixed(t, d)])
    return [d for case in tqdm(cases, desc=cfg["name"])
              for d in run_one_case(case, cfg, index)]

for cfg in CONFIGS:
    rows = run_config(cfg)
    json.dump({"config": cfg, "rows": rows},
              open(f"data/results/{cfg['name']}.json", "w"), indent=2)
    print(cfg["name"], len(rows), "decisions")
